# W9 · Day 1 — Hybrid Search: BM25 + Dense + RRF

**~75 minutes · in-class demo · Jupyter notebook · Track A**

**⚠ DANGER ZONE week.** Three new ideas stack this week: hybrid search
(today), reranking (tomorrow), and query rewriting (Stretch lab). We
deliberately move slower and split the hands-on into two shorter blocks
rather than one long one. If Day 1 feels dense — normal. Reinforcement
happens through the lab, not through more content.

**Today (Day 1):** name the naive RAG failures, meet BM25, learn RRF
(Reciprocal Rank Fusion), build a `hybrid_retrieve` function.

**Tomorrow (Day 2):** rerankers, query rewriting, LlamaIndex intro.

**Corpus:** 15 short docs about a fictional 'Acme Analytics Platform' —
features, error codes, pricing tiers, API endpoints. Designed to expose
the tension between BM25 (exact match) and dense retrieval (semantic).

**Cost per full run:** ~$0.005 (embedding calls only).

**Notebook flow:**
- Cell 1: Setup + corpus load
- Cell 2: The dense-only failure demo — run 4 queries against Qdrant, see 2 fail
- Cell 3: Failure taxonomy — 4 modes mapped to upgrades
- Cells 4-5: BM25 fundamentals + BM25 in code
- Cell 6: Neither dominates — comparison table
- Cell 7: RRF explained — the formula, why k=60
- Cell 8: Hybrid in code — combine BM25 + Dense via RRF
- Cell 9: Wrap + Day 2 homework (cross-encoder pre-download)

---

## Cell 1 — Setup

Confirm env, load corpus, connect to Qdrant. The corpus is 15 short docs
generated by `generate_acme_corpus.py`. If it's missing, run that script
first.

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook"
assert os.environ.get("QDRANT_URL"),     "Set QDRANT_URL — from W7 Qdrant Cloud setup"
assert os.environ.get("QDRANT_API_KEY"), "Set QDRANT_API_KEY — from W7 Qdrant Cloud setup"

# Add cwd to path so we can import the helper module
sys.path.insert(0, str(Path.cwd()))

# Load the corpus (15 docs) and 6 test queries with expected-winner annotations
CORPUS_PATH  = Path("sample_docs/acme_docs.jsonl")
QUERIES_PATH = Path("sample_docs/acme_test_queries.jsonl")

assert CORPUS_PATH.exists(), f"Missing {CORPUS_PATH}. Run: python demos/generate_acme_corpus.py"
assert QUERIES_PATH.exists(), f"Missing {QUERIES_PATH}. Run: python demos/generate_acme_corpus.py"

corpus  = [json.loads(line) for line in CORPUS_PATH.read_text().splitlines() if line]
queries = [json.loads(line) for line in QUERIES_PATH.read_text().splitlines() if line]

print(f"Loaded {len(corpus)} docs from {CORPUS_PATH.name}:")
from collections import Counter
for cat, n in Counter(d['category'] for d in corpus).items():
    print(f"  {cat:10s}  {n} docs")
print()
print(f"Loaded {len(queries)} test queries:")
for q in queries:
    print(f"  ({q['expected_winner']:5s})  {q['q']}")

**Notice the 'expected_winner' column.** For each query, we've annotated
which retriever *should* dominate:
- **BM25** wins on exact-match queries (error codes, API endpoint paths)
- **DENSE** wins on synonym-heavy queries (paraphrases, conversational phrasing)

The whole point of hybrid retrieval is: **you don't know in advance which
kind of query a user will type.** Combining both retrievers means either
type gets a good result.

---

## Cell 2 — The dense-only failure demo

First, embed the corpus and push to Qdrant. Then run the 6 test queries
against pure dense retrieval and see where it fails.

**Prediction:** dense will succeed on the DENSE queries (synonyms are its
strength) and fail or rank low on the BM25 queries (exact codes and
endpoint paths).

In [ ]:
from openai import OpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

openai_client = OpenAI()
qdrant = QdrantClient(url=os.environ["QDRANT_URL"], api_key=os.environ["QDRANT_API_KEY"])

COLLECTION = "wk09_day1_acme"

# Recreate — makes this cell re-runnable
try:
    qdrant.delete_collection(COLLECTION)
except Exception:
    pass

qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

# Embed all 15 docs in one API call
print("Embedding 15 Acme docs...")
texts = [d["text"] for d in corpus]
resp = openai_client.embeddings.create(model="text-embedding-3-small", input=texts)
vectors = [item.embedding for item in resp.data]

# Upsert
points = [
    PointStruct(id=idx, vector=vec, payload=doc)
    for idx, (doc, vec) in enumerate(zip(corpus, vectors))
]
qdrant.upsert(collection_name=COLLECTION, points=points)
print(f"Upserted {len(points)} vectors into {COLLECTION!r}.")

In [ ]:
def dense_search(query: str, k: int = 3) -> list[dict]:
    """Query Qdrant for top-K by cosine similarity."""
    q_vec = openai_client.embeddings.create(
        model="text-embedding-3-small", input=[query]
    ).data[0].embedding
    hits = qdrant.query_points(collection_name=COLLECTION, query=q_vec, limit=k).points
    return [
        {"id": h.payload["id"], "title": h.payload["title"], "score": h.score, "doc": h.payload}
        for h in hits
    ]

print(f"══ Dense retrieval on all 6 queries (top-1) ══\n")
print(f"  {'Expected':10s}  {'Query':<50s}  {'Retrieved':<20s}  Verdict")
print(f"  {'--------':10s}  {'-----':<50s}  {'---------':<20s}  -------")

dense_correct = 0
for tq in queries:
    hits = dense_search(tq["q"], k=1)
    got = hits[0]["id"]
    correct = got == tq["expected_id"]
    if correct: dense_correct += 1
    verdict = "✓" if correct else "✗ (expected " + tq["expected_id"] + ")"
    print(f"  {tq['expected_winner']:10s}  {tq['q'][:50]:<50s}  {got:<20s}  {verdict}")

print(f"\nDense-only: {dense_correct}/{len(queries)} correct at top-1")

**Discussion moment (the whole reason we're here):**
- On which queries did dense succeed? What do they have in common?
- On which queries did dense fail? What do those have in common?
- Notice the failures cluster on queries with **exact codes** (AC-1042) and
  **exact endpoint paths** (/v2/dashboards). Dense embeddings compress these
  into meaning space — but the meaning of a code is 'the code itself.'
  Meaning-space compression loses the surface form.

**This is the core failure mode BM25 fixes.** Next few cells build it.

---

## Cell 3 — Failure taxonomy: 4 modes → 4 upgrades

The RAG failures you've been seeing since W6 aren't random. They cluster
into four patterns. Each has a specific fix:

| Failure mode | Example | Fix |
|---|---|---|
| **Missed exact match** | 'what is error AC-1042' misses the AC-1042 doc | **Hybrid (BM25 + Dense)** — today, Day 1 |
| **Missed synonym** | 'notified when things break' misses the alerting doc | **Hybrid** — also today, other direction |
| **Noisy top-K** | correct answer is at rank 5, not rank 1 | **Reranker** — tomorrow, Day 2 |
| **Ambiguous query** | 'the dashboard' — which dashboard? | **Query rewriting** — Stretch lab |

**Today's job:** fix the first two by adding BM25 alongside dense.

**Tomorrow's job:** fix the third by adding a cross-encoder reranker on
top of the hybrid results.

**Take-home Stretch:** try one query rewriting technique for the fourth.

---

## Cell 4 — BM25 fundamentals

**BM25** = Best Match 25 (the 25th ranking function from the Okapi
information retrieval project, 1994). It's a **scoring function** — given a
query and a document, it returns a relevance score. Higher = more relevant.

**Three components:**

1. **TF (Term Frequency)** — how often the query term appears in the doc.
   More matches = higher score. But with **saturation**: the 10th match
   isn't 10× as important as the 1st.

2. **IDF (Inverse Document Frequency)** — how rare the query term is across
   the whole corpus. Rare terms weight higher (`AC-1042` appearing in 1 doc
   is more informative than `the` appearing in all 15).

3. **Length normalisation** — long docs shouldn't automatically outscore
   short docs just because they have room for more matches.

**Simplified formula:**
```
score(doc, query) = sum over terms in query of:
    IDF(term) × TF_saturated(term, doc) × length_norm(doc)
```

**Why not raw TF-IDF?** TF-IDF has no saturation and no length norm — a
long doc with 20 matches beats a short doc with 5 matches, even if the
short doc is more focused. BM25 fixes both.

---

## Cell 5 — BM25 in code

The `rank-bm25` library gives you `BM25Okapi` — 3 lines to fit, 1 line to
query.

**Critical detail: tokenization.** BM25 works on tokens, not raw text. How
you tokenize decides what counts as a match. If `AC-1042` gets split into
`['AC', '1042']`, BM25 will treat them as separate terms — and the exact-
code advantage disappears.

In [ ]:
import re
from rank_bm25 import BM25Okapi

def simple_tokenize(text: str) -> list[str]:
    """Lowercase + word-and-alphanumeric split, KEEPING hyphens/slashes inside tokens.
    
    Critical: this pattern preserves 'ac-1042' and 'v2/dashboards' as single
    tokens rather than splitting them. That's what makes BM25 catch exact IDs.
    """
    return re.findall(r'[a-z0-9][a-z0-9\-/_]*', text.lower())

# Tokenization sanity check — critical to inspect before trusting BM25
print("══ Tokenization checks ══")
for text in ["Error AC-1042 indicates a timeout", "POST /v2/dashboards endpoint",
             "the enterprise plan cost"]:
    tokens = simple_tokenize(text)
    print(f"  {text!r:40s} → {tokens}")

# Fit BM25
print(f"\n══ Building BM25 index ══")
tokenized_corpus = [simple_tokenize(d["text"] + " " + d["title"]) for d in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print(f"Indexed {len(corpus)} docs. Ready to query.")

In [ ]:
def bm25_search(query: str, k: int = 3) -> list[dict]:
    """Query BM25 index; return top-K."""
    tokens = simple_tokenize(query)
    scores = bm25.get_scores(tokens)
    ranked = sorted(zip(scores, corpus), key=lambda pair: pair[0], reverse=True)
    return [
        {"id": doc["id"], "title": doc["title"], "score": float(score), "doc": doc}
        for score, doc in ranked[:k]
    ]

print(f"══ BM25 retrieval on all 6 queries (top-1) ══\n")
print(f"  {'Expected':10s}  {'Query':<50s}  {'Retrieved':<20s}  Verdict")
print(f"  {'--------':10s}  {'-----':<50s}  {'---------':<20s}  -------")

bm25_correct = 0
for tq in queries:
    hits = bm25_search(tq["q"], k=1)
    got = hits[0]["id"]
    correct = got == tq["expected_id"]
    if correct: bm25_correct += 1
    verdict = "✓" if correct else "✗ (expected " + tq["expected_id"] + ")"
    print(f"  {tq['expected_winner']:10s}  {tq['q'][:50]:<50s}  {got:<20s}  {verdict}")

print(f"\nBM25-only: {bm25_correct}/{len(queries)} correct at top-1")

**Discussion moment — the real-world messiness:**
- Did BM25 win all the queries marked BM25? If some failed, why?
- Look closely: on 'error code AC-1042', BM25 may have returned `err_ac4408`
  instead of `err_ac1042`. Read `err_ac4408.text` — it **references** AC-1042
  in its resolution steps. BM25 sees the token `ac-1042` in both docs and
  can pick the wrong one when other query terms tip the balance.
- This is a **legitimate real-world failure mode**: documentation often
  cross-references other entries by ID. BM25 alone can't tell 'the doc
  ABOUT AC-1042' from 'the doc THAT MENTIONS AC-1042'.
- **Dense might get this right** — it can pick up on 'this doc is about
  AC-1042' as a semantic property. Hybrid combines the strengths.

---

## Cell 6 — Neither dominates: side-by-side comparison

Now put dense and BM25 next to each other for all 6 queries. This is the
moment where 'we need both' becomes visible.

In [ ]:
print(f"══ Dense vs BM25 side-by-side (top-1 each) ══\n")
print(f"  {'Query':<50s}  {'DENSE top-1':<20s}  {'BM25 top-1':<20s}  {'Correct':<10s}")
print(f"  {'-----':<50s}  {'-----------':<20s}  {'----------':<20s}  {'-------':<10s}")

dense_wins, bm25_wins, both_win, both_fail = 0, 0, 0, 0

for tq in queries:
    dense_top = dense_search(tq["q"], k=1)[0]
    bm25_top  = bm25_search(tq["q"], k=1)[0]
    
    dense_ok = dense_top["id"] == tq["expected_id"]
    bm25_ok  = bm25_top["id"]  == tq["expected_id"]
    
    if dense_ok and bm25_ok:     tag = "both ✓";     both_win += 1
    elif dense_ok:               tag = "dense only";  dense_wins += 1
    elif bm25_ok:                tag = "bm25 only";   bm25_wins += 1
    else:                        tag = "both ✗";     both_fail += 1
    
    d_mark = "✓" if dense_ok else " "
    b_mark = "✓" if bm25_ok else " "
    print(f"  {tq['q'][:50]:<50s}  {d_mark} {dense_top['id']:<18s}  {b_mark} {bm25_top['id']:<18s}  {tag}")

print()
print(f"Both right:         {both_win}/{len(queries)}")
print(f"Only dense right:   {dense_wins}/{len(queries)}")
print(f"Only BM25 right:    {bm25_wins}/{len(queries)}")
print(f"Both wrong:         {both_fail}/{len(queries)}")

**The killer observation:** count the 'only dense right' + 'only BM25 right'
rows. That's the population of queries where **combining the two would give
you an answer that neither alone gets to top-1.**

For 'both wrong' rows — even hybrid may not save you. Those are candidates
for the reranker (Day 2) or query rewriting (Stretch lab).

**Rule of thumb from experience:** on enterprise corpora (docs, policies,
product manuals), hybrid retrieval typically lifts precision@3 by 10-20
percentage points over pure dense. On purely conversational corpora (chat
logs, blog posts), the lift is smaller — dense already handles the
language well.

---

## Cell 7 — RRF: how to combine two ranked lists

You have two ranked lists (BM25 top-K, dense top-K). You need ONE final
ranked list. Options:

1. **Score sum** — add BM25 score + dense score. **Broken.** BM25 scores
   are unbounded (~5-40), cosine is [-1,1]. One retriever completely
   dominates.

2. **Weighted score sum** — normalize each score to [0,1] then combine
   with weights (e.g. 0.5·bm25 + 0.5·dense). Works but requires tuning
   weights per corpus.

3. **RRF (Reciprocal Rank Fusion)** — combine by **rank position only**,
   ignoring the raw scores. No tuning needed. This is the industry
   standard.

**RRF formula (Cormack et al., 2009):**
```
score(doc) = sum over ranked_lists of:  1 / (k + rank_in_list)
```
where `k=60` is the standard smoothing constant.

**Why k=60?** Small enough that rank still matters (rank-1 scores much
higher than rank-50), large enough that top-1 doesn't massively dominate
top-2 (which would let a single retriever's top-pick always win).

**Intuition:** appearing high in BOTH lists gets you a boost. Appearing
only in one list keeps you in contention. Appearing low in one list barely
helps at all.

In [ ]:
def rrf_fuse(ranked_lists: list[list[dict]], k: int = 60, top_n: int = 5) -> list[dict]:
    """Fuse multiple ranked lists via Reciprocal Rank Fusion.
    
    Each entry in ranked_lists is a list of dicts with an 'id' key.
    Returns the top-N fused results (with 'rrf_score' added).
    """
    scores = {}
    docs   = {}
    for ranked in ranked_lists:
        for rank, hit in enumerate(ranked, start=1):
            doc_id = hit["id"]
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
            if doc_id not in docs:
                docs[doc_id] = hit
    
    fused = sorted(scores.items(), key=lambda p: p[1], reverse=True)[:top_n]
    return [{**docs[doc_id], "rrf_score": score} for doc_id, score in fused]

# Toy example to show RRF math
print("══ RRF sanity check ══")
list_bm25  = [{"id": "doc_A"}, {"id": "doc_B"}, {"id": "doc_C"}]
list_dense = [{"id": "doc_C"}, {"id": "doc_A"}, {"id": "doc_D"}]

print(f"  BM25 ranks:  {[h['id'] for h in list_bm25]}")
print(f"  Dense ranks: {[h['id'] for h in list_dense]}")

fused = rrf_fuse([list_bm25, list_dense], k=60, top_n=5)
print(f"\n  RRF fused (k=60):")
for h in fused:
    print(f"    {h['id']}  rrf_score = {h['rrf_score']:.5f}")

print(f"\n  Interpretation:")
print(f"  - doc_A ranks 1st and 2nd → score = 1/61 + 1/62 = {1/61 + 1/62:.5f}")
print(f"  - doc_C ranks 3rd and 1st → score = 1/63 + 1/61 = {1/63 + 1/61:.5f}")
print(f"  - doc_B appears in one list only → score = 1/62 = {1/62:.5f}")
print(f"  - doc_D appears in one list only → score = 1/63 = {1/63:.5f}")

**Notice how appearing in BOTH lists (A and C) beats appearing in only one
(B and D), even when the sole appearance is at rank 1.** That's RRF's
consensus behavior — it rewards docs that both retrievers agree on.

---

## Cell 8 — Hybrid retrieve: put it all together

Now the payoff. Wrap dense + BM25 + RRF into `hybrid_retrieve()` and run
on all 6 queries. Compare against pure dense and pure BM25.

In [ ]:
def hybrid_retrieve(query: str, k_per_retriever: int = 10, k_final: int = 3) -> list[dict]:
    """BM25 + Dense + RRF combined retrieval.
    
    k_per_retriever: how many to take from each retriever before fusion (10 is standard)
    k_final:         how many to return after RRF fusion
    """
    bm25_hits  = bm25_search(query, k=k_per_retriever)
    dense_hits = dense_search(query, k=k_per_retriever)
    return rrf_fuse([bm25_hits, dense_hits], k=60, top_n=k_final)

print(f"══ Hybrid retrieval on all 6 queries (top-1) ══\n")
print(f"  {'Query':<50s}  {'Hybrid top-1':<20s}  {'Verdict'}")
print(f"  {'-----':<50s}  {'------------':<20s}  {'-------'}")

hybrid_correct = 0
for tq in queries:
    hits = hybrid_retrieve(tq["q"], k_per_retriever=10, k_final=1)
    got = hits[0]["id"]
    correct = got == tq["expected_id"]
    if correct: hybrid_correct += 1
    verdict = "✓" if correct else "✗ (expected " + tq["expected_id"] + ")"
    print(f"  {tq['q'][:50]:<50s}  {got:<20s}  {verdict}")

print()
print(f"══ Summary across the 3 retrievers ══")
print(f"  Dense only:  {dense_correct}/{len(queries)}")
print(f"  BM25 only:   {bm25_correct}/{len(queries)}")
print(f"  Hybrid RRF:  {hybrid_correct}/{len(queries)}")

**This is the DANGER ZONE peak — the moment three concepts snap into place:**

1. Dense catches semantic matches (synonyms, paraphrases)
2. BM25 catches exact matches (codes, endpoints, proper nouns)
3. RRF combines them without needing to tune weights

**Expected on this small corpus:** hybrid gets ≥ max(dense, BM25) alone.
It may not perfect-score (some queries are also confused by cross-references
in the docs — that's the reranker's job tomorrow), but it should lift the
worst cases.

**Track B implication:** in your capstone you'll wire this same shape into
`src/rag/retrieval.py` and re-run your golden set. Expected lift on
precision@3: 5-20 percentage points.

---

## Cell 9 — Wrap + Day 2 homework

**Today (Day 1) you built:**
1. Named the naive-RAG failure modes and mapped each to an upgrade
2. Meet BM25 — TF + IDF + length norm, why not raw TF-IDF
3. Saw where BM25 wins (exact codes) and where dense wins (synonyms)
4. Learned RRF (Reciprocal Rank Fusion) — the industry-standard way to combine ranked lists
5. Wired `hybrid_retrieve` — BM25 + Dense + RRF in ~10 lines

**Tomorrow (Day 2) covers:**
- Cross-encoder rerankers — the second stage that reorders your top-10 into a precise top-3
- Query rewriting patterns — HyDE, multi-query, step-back (introduced; Stretch lab)
- LlamaIndex — brief framework intro (no adoption)

**Track B (take-home):** apply hybrid+rerank to YOUR capstone, measure
precision@k delta vs W8, update ADR. **This is the first Core+Stretch week**
— Core = hybrid+rerank on golden set (mandatory); Stretch = try one query
rewriting technique (optional).

---

## HOMEWORK BEFORE DAY 2 — pre-download the cross-encoder (~5 min)

**Do this before Day 2 starts.** The cross-encoder is ~80MB from HuggingFace.
You do not want to be waiting for a download during a live 90-min session
(especially on flaky Vocareum networks or corporate proxies).

**Run this once:**
```bash
pip install sentence-transformers
```

**Then in Python (in the Vocareum kernel you'll use for Day 2):**
```python
from sentence_transformers import CrossEncoder
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('Downloaded. Cached at:', reranker.model.config._name_or_path)
```

**Verify with a test scoring call:**
```python
print(reranker.predict([('what is dashboards', 'Dashboards are visualisations of data')]))
# Expected: array with one float, roughly in the range -5 to +10
```

If either step fails on Vocareum, tell your instructor **before** Day 2
— you'll need help resolving network/proxy issues that can't be sorted in
class. Day 2 Cell 1 hard-asserts the model is available.